In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Joints on the wire: the SO-101 server node, its controllers, and the arm client

The camera pattern, applied to the thing that can actually pinch you.
**Exactly one process owns the Feetech serial bus**: the server node. It is
the hardware side of **ros2_control**: the controllers sit between it and
every client, and turn a move into a smooth trajectory.

![Alt Text](../../animations/SO-101_Loop_Animation.gif)

**This notebook moves the real arm.** Start it with `./run_pipeline.sh`
(not `--digital-twin`). That script starts `python -m so101_sim.stack`
with `digital-twin false`, which owns the Feetech bus and its controllers.
This notebook only connects, with the same `Ros2ArmClient` the workshop
uses against that launch. It does not open the bus itself.

The conventions that make this safe (and your TODOs implement):

- behaviors speak **degrees**, keyed `shoulder_pan, shoulder_lift,
  elbow_flex, wrist_flex, wrist_roll, gripper`; ROS speaks **radians**, and
  the joint map converts at the server's and the client's topics;
  **partial commands are legal**;
- every command passes `clamp_joints`, on the client *and* the server
  (belt and braces);
- the state's `effort` carries the **gripper's torque in N·m**, from its
  servo current, with **NaN meaning "not readable"**. That feedback lets
  the grip behavior stop squeezing at first contact instead of crushing
  whatever it holds.

**Matching TODOs:** `2.4`-`2.9` in `project/ros/arm_node.py` and
`project/ros/arm_client.py`.

In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import threading, time
import math
import queue
import tempfile

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
# run_pipeline.sh starts in this repo, so `import vla_pipeline` works there.
# This kernel starts in the notebook directory.
REPO = PROJECT.parents[1]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
from common.motion import JOINT_LIMITS, MOTOR_NAMES, REST_POSE, clamp_joints
from common.ros_helpers import Spinner

cfg = load_config()
print("running against:", PROJECT)

In [ ]:
# ROS 2 check: this notebook runs on real ROS 2 (rclpy) only.
try:
    import rclpy

    HAVE_ROS = True
except ImportError:
    HAVE_ROS = False

ROS_HELP = """rclpy is not importable in this kernel, so the ROS 2 cells cannot run."""


def require_ros():
    if not HAVE_ROS:
        raise RuntimeError(ROS_HELP)


if HAVE_ROS:
    if not rclpy.ok():
        rclpy.init()
    print("real ROS 2 (rclpy) ready")
else:
    print(ROS_HELP)

> **About the arm.** `./run_pipeline.sh` already started it (leave off
> `--digital-twin`). This notebook only connects. Only one arm server and
> one set of controllers may run at a time.

In [ ]:
require_ros()

# One background thread spins the notebook's nodes, so the notebook stays
# interactive while they tick. (The pipeline's orchestrator embeds its
# client nodes the same way.)
host = Spinner("demo_host")

## 1. A minimal arm server: ros2_control's hardware side

The server is not started here. `./run_pipeline.sh` already started it
with these commands (`digital-twin false` is the real bus):

`python -m so101_sim.stack --arm false --cameras true --digital-twin false --dry-run false`

`python -m so101_sim.stack --arm true --cameras false --digital-twin false --dry-run false`

That process owns the bus. The implementation of TODOs 2.4-2.6 lives in
`solution/ros/arm_node.py`.

The server never talks to behaviors. It talks to `topic_based_ros2_control`,
the controllers' hardware plugin: setpoints arrive on
`/so101/hardware/joint_commands` and the arm's state goes back on
`/so101/hardware/joint_states`, both `sensor_msgs/JointState` in
**radians**. The joint map (`so101_sim.mapping.JointMap`) converts to and
from the bus's degrees. The state also carries each joint's velocity, from
the change in position since the last tick, and the gripper's torque in
N·m (NaN when the bus can't read it).

Note the **control thread**: subscriptions land on executor threads, but
only one sequential loop touches the arm. That loop is the only thing on
the serial bus. No contention, ever. It writes every fresh setpoint as it
comes - the controllers interpolate the motion - through LeRobot's per-tick
clamp, the transport's own backstop.

In [ ]:
require_ros()

# The arm server is the so101_sim.stack process ./run_pipeline.sh already
# started, with digital-twin false. That process owns the Feetech bus.
print("arm server is the one ./run_pipeline.sh started")


## 2. The controllers in between

`launch/so101_control.launch.py` starts ros2_control on top of the server:

- **`topic_based_ros2_control`**, the hardware plugin: it publishes the
  controllers' setpoints to the server and reads the server's state back
  (section 1's two topics). It only publishes while the arm is away from
  the setpoint, so a still arm costs no traffic;
- **`arm_controller`** (a `JointTrajectoryController`): a move is one
  `FollowJointTrajectory` goal - where to go, and how long the servos'
  profile says it takes. The controller interpolates it, then reports
  whether the joints got there within the goal's tolerance;
- **`arm_forward_controller`** and **`gripper_forward_controller`**:
  setpoints passed straight through, for streamed motion (gesture mimic,
  person tracking) and for the gripper;
- **`joint_state_broadcaster`**: the arm's state for everyone, on
  `/joint_states`.

Those controllers are already running inside the same stack. This notebook
does not start `launch/so101_control.launch.py`.

In [ ]:
require_ros()
print("controllers are the ones ./run_pipeline.sh already started")

## 3. The client side, and the first rule of robot clients

**Never command an arm whose state you can't see.** The client blocks until
`/joint_states` and the controllers are live, *then* exposes the `ArmClient`
API. That part, and running a move as one trajectory goal, comes from the
provided base, `common.ros2_control_arm.ControllerArm`. A client wires the
endpoints and the two conversions - exactly TODOs 2.7-2.9:

- the endpoints: the trajectory action client, the forward controllers'
  publishers, and the `/joint_states` subscription;
- a streamed setpoint: clamped, then sent in radians;
- a state: radians back to degrees, and the gripper's effort with NaN as
  `None`.

In [ ]:
require_ros()

# The workshop client for the launch ./run_pipeline.sh started.
from ros.arm_client import Ros2ArmClient

arm = Ros2ArmClient(cfg)
print(
    "state stream and controllers live. rest pose:",
    {k: round(v, 1) for k, v in arm.get_joints().items()},
)
print(
    "gripper effort:",
    arm.get_gripper_load(),
    " (Present_Current counts, from the N·m on the wire; None means not readable, NOT zero)",
)


## 4. Drive every joint by hand

Before scripting motion, get a feel for the real arm. One slider per
joint, ranges straight from `JOINT_LIMITS`. Drag a slider and **release**
it: the release value becomes one move - one trajectory goal, timed by the
servos' own profile - and the controller drives the arm along it. One
gesture, one goal. (Why on release? A running cell only receives a drag's
flood of widget syncs throttled, while one sync per gesture always lands.)
A move runs to its end; a gesture released meanwhile waits, and only the
newest waiting one runs.

The sliders start at `REST_POSE` (wrist_flex rests at 22.5 on this rig)
and the cell sends the arm there first, so the UI and the metal agree from
the first pixel. The gripper slider spans the jaws' calibrated range,
43 (closed) to 95 (open).

Two extras:

- **link shoulder_lift + elbow_flex** (on by default): the two proximal
  joints move together, in opposition - move shoulder_lift by +0.1 and
  elbow_flex follows by -0.1, in the SAME goal - which keeps the forearm
  attitude roughly steady as the shoulder moves. Untick to drive them
  independently.
- **rest pose** sends ONE full-pose goal, then quietly syncs the sliders
  to match.

The loop below just keeps UI events flowing and streams the *measured*
joint state back under the sliders. Press **stop** to end the loop and
move to the next cell.

These sliders move the physical arm. Move them gently and keep clear.

In [ ]:
require_ros()

LINKED = ("shoulder_lift", "elbow_flex")  # move together, in opposition

# One worker runs the moves, one at a time; the UI only queues them. The
# queue holds one: a newer gesture replaces an older one that has not started.
goals = queue.Queue(maxsize=1)
DONE = object()


def submit(goal):
    try:
        goals.get_nowait()  # drop the older gesture that has not started
    except queue.Empty:
        pass
    goals.put_nowait(goal)


def mover():
    while (goal := goals.get()) is not DONE:
        arm.move_to(goal)


worker = threading.Thread(target=mover, daemon=True)
worker.start()
submit(dict(REST_POSE))  # start from a known pose: rest - the sliders below default to it too

sliders = {}
for name in MOTOR_NAMES:
    lo, hi = JOINT_LIMITS[name]
    sliders[name] = widgets.FloatSlider(
        value=REST_POSE[name],
        min=lo,
        max=hi,
        step=0.1 if name in LINKED else 0.5,
        description=name,
        continuous_update=False,  # one state sync per gesture, on release
        readout_format=".1f",
        layout=widgets.Layout(width="440px"),
        style={"description_width": "110px"},
    )

link_chk = widgets.Checkbox(
    value=True,
    indent=False,
    description="link shoulder_lift + elbow_flex (opposite)",
    layout=widgets.Layout(width="340px"),
)

# Event-driven goals: each slider CHANGE queues one partial goal; the
# controller holds every joint it leaves out. With the link on, a
# shoulder_lift move of +d takes elbow_flex along by -d (and vice versa) in
# the SAME goal, so the pair stays coordinated.
syncing = {"on": False}  # guards programmatic slider writes


def make_sender(name):
    def _send(change):
        if syncing["on"]:
            return
        cmd = {name: float(change["new"])}
        if link_chk.value and name in LINKED:
            other = LINKED[1] if name == LINKED[0] else LINKED[0]
            delta = float(change["new"]) - float(change["old"])
            partner = sliders[other]
            target = max(partner.min, min(partner.max, partner.value - delta))
            syncing["on"] = True
            try:
                partner.value = target  # mirror the UI without re-sending
            finally:
                syncing["on"] = False
            cmd[other] = float(target)
        submit(cmd)  # ONE goal for the whole gesture

    return _send


senders = {name: make_sender(name) for name in sliders}
for name, s in sliders.items():
    s.observe(senders[name], "value")

stop_btn = widgets.Button(description="stop", button_style="danger")
rest_btn = widgets.Button(description="rest pose", button_style="warning")
stopped = {"flag": False}
stop_btn.on_click(lambda _: stopped.update(flag=True))


def go_rest(_):
    submit(dict(REST_POSE))  # ONE full-pose goal...
    syncing["on"] = True  # ...then sync the UI quietly
    try:
        for n in MOTOR_NAMES:
            sliders[n].value = REST_POSE[n]
    finally:
        syncing["on"] = False


rest_btn.on_click(go_rest)

readout = widgets.Label()
display(
    widgets.VBox(
        [*sliders.values(), link_chk, widgets.HBox([rest_btn, stop_btn]), readout]
    )
)

# The loop's only jobs: deliver the widget events (poll) and stream the
# MEASURED state back so you can watch the arm track each goal.
with ui_events() as poll:
    while not stopped["flag"]:
        poll(10)  # slider releases + clicks land here
        j = arm.get_joints()
        readout.value = "state:  " + "   ".join(f"{n}={j[n]:.1f}" for n in MOTOR_NAMES)
        time.sleep(0.03)

for name, s in sliders.items():  # detach: the sliders go quiet
    s.unobserve(senders[name], "value")  # once you leave this cell
submit(DONE)  # the worker finishes the move it runs, then ends
worker.join()
print("joint control loop stopped - sliders detached")

## 5. The safety clamp survives the transport

`clamp_joints` runs on both ends. Command something insane and watch physics
stay bounded. Heads up: the arm really will swing to its +86 degree pan
limit and back (the software clamp - the mechanical range pulled in 25% so
the arm clears the environment), so keep the space around it open anyway.
`move_to` returns when the controller reports the move over: `"arrived"`
(within the goal's tolerance), `"blocked"` (it ended short) or `"stopped"`.

In [ ]:
require_ros()

pan_lo, pan_hi = JOINT_LIMITS["shoulder_pan"]
how = arm.move_to({"shoulder_pan": 500.0})  # 500 degrees into an +/-86 degree clamp
pan = arm.get_joints()["shoulder_pan"]
print(f"commanded 500 -> the move {how} at {pan:.1f}  (limit: {pan_lo:+.0f}/{pan_hi:+.0f})")
arm.move_to({"shoulder_pan": 0.0})

## 6. Why the effort field exists

The gripper servo reports its **Present_Current** through the bus. The
server turns it into torque (N·m, along the STS3215 datasheet line) and
sends it in `JointState.effort`, sampled every few control ticks; the
client turns it back into current counts, so the grip behavior keeps its
thresholds. Watching current instead of commanded position is what lets a
grip behavior stop closing at first sustained contact rather than driving
to a target and crushing the object on the way.

Two conventions make it work over any transport, and you implement both:
NaN on the wire means "not readable" and the client maps it to `None`,
never zero; when effort is unavailable, behaviors fall back to detecting a
position stall instead.

## 7. On the rig

```bash
# Terminal 1 - the same two commands as ./run_pipeline.sh (real arm):
python -m so101_sim.stack --arm false --cameras true --digital-twin false --dry-run false
python -m so101_sim.stack --arm true --cameras false --digital-twin false --dry-run false

# Terminal 2 - watch it like an operator:
ros2 control list_controllers          # which controllers run
ros2 topic echo /joint_states          # positions (radians) + effort[gripper] (N·m)

# Terminal 3 - your client's round-trip test:
python -m ros.arm_client
```

Real-hardware notes (from the pipeline README): calibrate once in a
foreground terminal before backgrounding the server; check `motor_port`
permissions; the server parks the arm at `REST_POSE` on any shutdown path.

## 8. Shut down

Close the client. The arm launch keeps running; `./run_pipeline.sh` owns it.

In [ ]:
require_ros()

arm.close()
host.close()
print("arm client closed; ./run_pipeline.sh keeps the arm running")

## 9. Your turn

Open **`project/ros/arm_node.py`** and **`project/ros/arm_client.py`**:

| TODO | What | Taught in |
|---|---|---|
| **2.4** | server `_setup_ros`: cmd sub + state pub | section 1 |
| **2.5** | server `_on_command`: radians to degrees, filter, clamp, store | sections 1 and 5 |
| **2.6** | server `_publish_state`: radians + velocity + effort (N·m)/NaN | sections 1 and 6 |
| **2.7** | client `_setup_ros`: trajectory action, forward controllers, `/joint_states` | sections 2 and 3 |
| **2.8** | client `send_joints`: clamp + stream in radians | sections 3 and 5 |
| **2.9** | client `_on_state`: radians to degrees + effort->None | sections 3 and 6 |


Then, from the `project/` directory:


```bash
python -m ros.selftest

# to see all the TODOs you have left
python app.py
```



Next: `03_intent_node.ipynb`, language becomes a node.